In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/refactored_code"))
import yaml
from sklearn.model_selection import train_test_split



In [4]:
from src.utils.bigwig_utils import get_preprocessed_encode_cpg_dataframe
# from src.utils.formatting import create_length_window_id, combine_cpg_dfs, group_methyl_dfs, combine_seperated_dfs_to_train_test
from src.data_extractor import create_cpg_window_files, extract_data

In [ ]:


def main():
    cfg = yaml.safe_load(open("../configs/config_extraction_intermidiate_unique_regions_split.yaml"))
    extract_data(cfg)


main()


Processing chromosome: chr1 with length: 248956422


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fb412d68b90>>
Traceback (most recent call last):
  File "/sci/nosnap/michall/roeizucker/new_python_env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
cfg = yaml.safe_load(open("../config_extraction_intermidiate_unique_regions_split.yaml"))
paths = cfg["paths"]
task = cfg["task"]
raw_files_paths = paths["raw_data_paths"]
chroms = task.get("chromosomes", None)
full_position_column_name = "full_position"
test_size = task.get("test_size", 0.2)
random_state = cfg.get("random_state", 42)

dfs = load_dfs(raw_files_paths, chroms, full_position_column_name)
labels = []  # same length/order as `kept`
for i in range(len(dfs)):
    labels.append(f"ind_{i}")

df = combine_cpg_dfs(full_position_column_name, dfs, labels)
# create_window_id(df1,5400)
grouped_df = group_methyl_dfs(dfs, df)

high_diff_quantile_value = grouped_df["high_diff"].quantile(0.95)
print("high_diff_quantile_value",high_diff_quantile_value)
curr_df = grouped_df.reset_index()
group_a_df = curr_df[curr_df["high_diff"] > high_diff_quantile_value]
group_b_df = curr_df[curr_df["high_diff"] <= high_diff_quantile_value]

train_df_group_a, test_df_group_a = train_test_split(
        group_a_df,
        test_size   = test_size,
        random_state = random_state,
    )

train_df_group_b, test_df_group_b = train_test_split(
        group_b_df,
        test_size   = test_size,
        random_state = random_state,
    )
train_df = pd.concat([train_df_group_a,train_df_group_b])
test_df = pd.concat([test_df_group_a,test_df_group_b])
train_path = save_cpg_dfs(cfg, train_df, test_df)
print(train_path)
# test_size = task.get("test_size", 0.2)
# random_state = cfg.get("random_state", 42)
# train_df_high_diff, test_df_high_diff = train_test_split(
#         df_high_diff,
#         test_size   = test_size,
#         random_state = random_state,
#     )

# train_df_low_diff, test_df_low_diff = train_test_split(
#         df_low_diff,
#         test_size   = test_size,
#         random_state = random_state,
#     )

Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167
Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167
Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167
count: 26472075
count: 26216604
count: 26289534
max_diff_quantile_value 13.526565551757812
high_diff_quantile_value 0.09627372841036565
/sci/archive/michall/roeizucker/Tom_project/created_datasets/selected_windows_train.csv


1. move code here to appropriate libraries
    a. moved some to formatting, check if need to return some, and move the rest to data_extractor
2. run with all chromosomes
3. change pipeline so that it's included.
4. do 3 types of training - all, jsut very variable, without very variable.
    a. train one on all and run on another person
    b. train one on without variable, retrain on only variable in another person, test on other person
    c. compare

In [ ]:
len(group_a_df),len(group_b_df)

(1105, 20992)

In [ ]:
assert False

In [ ]:
DIFF_THRESHOLD = 10 

In [72]:
df1 = get_preprocessed_encode_cpg_dataframe("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/downloaded_datasets/ENCFF479QRW.bigWig",["chr19","chr20"],verbose=True)
df2 = get_preprocessed_encode_cpg_dataframe("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/downloaded_datasets/ENCFF713LYH.bigWig",["chr19","chr20"],verbose=True)
df3 = get_preprocessed_encode_cpg_dataframe("/cs/usr/roeizucker/new_storage/jupyter_notebooks/Tom_Hope_Project/downloaded_datasets/ENCFF854OPR.bigWig",["chr19","chr20"],verbose=True)

description = [
    ("Homo sapiens adipose tissue", "ENCFF479QRW", "male adult (34 years)"),
    ("Homo sapiens adipose tissue", "ENCFF713LYH", "male child (3 years)"),
    ("Homo sapiens adipose tissue", "ENCFF854OPR", "female adult (30 years)")
]

Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167
Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167
Processing chromosome: chr19 with length: 58617616
Processing chromosome: chr20 with length: 64444167


In [ ]:
df1_small = df1.copy()
df2_small = df2.copy()


## basic statistics

ndividuals & coverage: #samples, median reads per CpG, #CpGs passing coverage filter in all samples.

Shared CpGs: count and % of CpGs shared across individuals after QC.

Variability (CpG-level): median and 95th-pct of |Δβ|; %CpGs with |Δβ| ≥ θ (e.g., 0.35).

Regions (VMRs): #regions, median length (bp), median CpGs/region, genome fraction covered.

Windows (5.4 kb)

In [2]:
def count_shared(individuals, positions):
    for df in individuals[1:]:
        positions = positions.intersection(set(df["full_position"]))
    return (len(positions))
def count_combined(individuals, positions):
    for df in individuals[1:]:
        positions = positions.union(set(df["full_position"]))
    return (len(positions))

In [ ]:
individuals = [df1, df2,df3]
for df in individuals:
    print("count:", len(df))
    df["full_position"] =  df["chrom"] + ":" + df["start"].astype(str)  + "-" + df["start"].astype(str)

positions = set(individuals[0]["full_position"])

shared_count = count_shared(individuals, positions)
combined_count = count_combined(individuals, positions)
print("shared", shared_count)
print("combined", combined_count)


count: 26472075
count: 26216604
count: 26289534
shared 25998993
combined 26561560


In [ ]:
full_position_column_name = "full_position"
labels = ["ind_A", "ind_B", "ind_C"]  # same length/order as `kept`
kept = [
    d for d in individuals
    if (full_position_column_name in d.columns) and ('methyl_rate' in d.columns) and d[full_position_column_name].notna().all()
]

out = pd.concat(
    [
        d[[full_position_column_name, 'methyl_rate']]
          .rename(columns={'methyl_rate': f'methyl_rate_{lab}'})
          .set_index(full_position_column_name)
        for lab, d in zip(labels, kept)
    ],
    axis=1, join='inner'
).reset_index()


In [ ]:
out2 = out.merge(
    kept[0][["full_position", "start","end","chrom"]].drop_duplicates("full_position"),
    on="full_position",
    how="left"
)


df = create_length_window_id(out2,5400)


In [ ]:

def group_regions():
    grouped = df1.groupby("window_id").agg({'start': list, 'end': list, 'methyl_rate': list,'chrom':lambda x: list(x[:1])[0],
                                        'bin_start':lambda x: list(x[:1])[0],'bin_end':lambda x: list(x[:1])[0]})
    val = grouped.reset_index().rename(columns={'start': 'starts', 'end': 'ends','methyl_rate':'values','bin_start':'start','bin_end':'end'})
    return val
df
val = group_regions()
print(val)


,full_position,methyl_rate_ind_0,methyl_rate_ind_1,methyl_rate_ind_2,start,end,chrom,0-1,0-2,1-2,max_diff,high_diff,bin_start,bin_end,window_id
0,chr19:70978-70978,0.0,0.0,0.0,70978,70979,chr19,0.0,0.0,0.0,0.0,False,70200,75600,chr19:70200-75600
1,chr19:70979-70979,0.0,0.0,0.0,70979,70980,chr19,0.0,0.0,0.0,0.0,False,70200,75600,chr19:70200-75600
2,chr19:70981-70981,0.0,0.0,0.0,70981,70982,chr19,0.0,0.0,0.0,0.0,False,70200,75600,chr19:70200-75600
3,chr19:70987-70987,0.0,0.0,0.0,70987,70988,chr19,0.0,0.0,0.0,0.0,False,70200,75600,chr19:70200-75600
4,chr19:70988-70988,0.0,0.0,0.0,70988,70989,chr19,0.0,0.0,0.0,0.0,False,70200,75600,chr19:70200-75600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25998988,chr20:64334087-64334087,0.0,0.0,0.0,64334087,64334088,chr20,0.0,0.0,0.0,0.0,False,64330200,64335600,chr20:64330200-64335600
25998989,chr20:64334089-64334089,0.0,0.0,0.0,64334089,64334090,chr20,0.0,0.0,0.0,0.0,False,64330200,64335600,chr20:64330200-64335600
25998990,chr20:64334093-64334093,0.0,0.0,0.0,64334093,64334094,chr20,0.0,0.0,0.0,0.0,False,64330200,64335600,chr20:64330200-64335600
25998991,chr20:64334095-64334095,0.0,0.0,0.0,64334095,64334096,chr20,0.0,0.0,0.0,0.0,False,64330200,64335600,chr20:64330200-64335600


In [ ]:
mcols = [c for c in out.columns if c.startswith("methyl_rate_")]
vals = out[mcols].astype(float).to_numpy()


In [ ]:
with pd.option_context('display.float_format', '{:,.2f}'.format):  # no sci, thousands sep
    print(out[mcols].var(axis=1).describe())

count   25,998,993.00
mean            19.30
std            131.43
min              0.00
25%              0.00
50%              0.00
75%              0.00
max          3,333.33
dtype: float64


In [ ]:
with pd.option_context('display.float_format', '{:,.2f}'.format):
    print(out[mcols].describe())

       methyl_rate_ind_A  methyl_rate_ind_B  methyl_rate_ind_C
count      25,998,993.00      25,998,993.00      25,998,993.00
mean                5.34               5.06               5.15
std                20.13              19.71              20.09
min                 0.00               0.00               0.00
25%                 0.00               0.00               0.00
50%                 0.00               0.00               0.00
75%                 0.00               0.00               0.00
max               100.00             100.00             100.00


In [ ]:
for col in mcols:
    print(col)
    print(out[col].value_counts(bins=10, sort=False))

methyl_rate_ind_A
(-0.101, 10.0]    80132498
(10.0, 20.0]        486492
(20.0, 30.0]        117797
(30.0, 40.0]         93120
(40.0, 50.0]        112914
(50.0, 60.0]        117608
(60.0, 70.0]        184486
(70.0, 80.0]        346769
(80.0, 90.0]        700251
(90.0, 100.0]      1809282
Name: count, dtype: int64
methyl_rate_ind_B
(-0.101, 10.0]    79869406
(10.0, 20.0]        795716
(20.0, 30.0]        140158
(30.0, 40.0]        134210
(40.0, 50.0]        169215
(50.0, 60.0]        143536
(60.0, 70.0]        227518
(70.0, 80.0]        390589
(80.0, 90.0]        573322
(90.0, 100.0]      1657547
Name: count, dtype: int64
methyl_rate_ind_C
(-0.101, 10.0]    80182079
(10.0, 20.0]        530897
(20.0, 30.0]         78569
(30.0, 40.0]         88286
(40.0, 50.0]        124708
(50.0, 60.0]        117527
(60.0, 70.0]        194809
(70.0, 80.0]        364085
(80.0, 90.0]        616934
(90.0, 100.0]      1803323
Name: count, dtype: int64


In [ ]:
# df

In [ ]:
# grouped metrics
# out
def temp_func():
    df = out2
    L = 5400  # window length

# if your coordinate is 'start' and already 0-based:
    return create_length_window_id(df, L)

def create_length_window_id(df, window_length):
    pos = df['start'].to_numpy(dtype='int64')
# if your 'start' is 1-based, convert first:
# pos = (df['start'].to_numpy(dtype='int64') - 1)
    bin_index = np.floor_divide(pos, window_length)          # integer division
    df['bin_start'] = (bin_index * window_length).astype('int64')
    df['bin_end']   = df['bin_start'] + window_length
# optional: stable ID (handy for joins/metrics)
    df['window_id'] = df['chrom'].astype('string') + ':' + df['bin_start'].astype(str) + '-' + df['bin_end'].astype(str)
    return df

df = temp_func()



In [ ]:
# create_window_id(df1,5400)
df["A-B"] = (df["methyl_rate_ind_A"] - df["methyl_rate_ind_B"]).abs()
df["A-C"] = (df["methyl_rate_ind_A"] - df["methyl_rate_ind_C"]).abs()
df["B-C"] = (df["methyl_rate_ind_B"] - df["methyl_rate_ind_C"]).abs()
df["max_diff"] = df[["A-B","A-C","B-C"]].max(axis=1)
df["high_diff"] = False
df.loc[df["max_diff"] > 15,"high_diff"] = True

In [ ]:

# def group_regions():
#     grouped = df1.groupby("window_id").agg({'start': list, 'end': list, 'methyl_rate': list,'chrom':lambda x: list(x[:1])[0],
#                                         'bin_start':lambda x: list(x[:1])[0],'bin_end':lambda x: list(x[:1])[0]})
#     val = grouped.reset_index().rename(columns={'start': 'starts', 'end': 'ends','methyl_rate':'values','bin_start':'start','bin_end':'end'})
#     return val
df
# val = group_regions()
# print(val)


,full_position,methyl_rate_ind_A,methyl_rate_ind_B,methyl_rate_ind_C,start,end,chrom,bin_start,bin_end,window_id,A-B,A-C,B-C,max_diff,high_diff
0,chr19:70978-70978,0.0,0.0,0.0,70978,70979,chr19,70200,75600,chr19:70200-75600,0.0,0.0,0.0,0.0,False
1,chr19:70979-70979,0.0,0.0,0.0,70979,70980,chr19,70200,75600,chr19:70200-75600,0.0,0.0,0.0,0.0,False
2,chr19:70981-70981,0.0,0.0,0.0,70981,70982,chr19,70200,75600,chr19:70200-75600,0.0,0.0,0.0,0.0,False
3,chr19:70987-70987,0.0,0.0,0.0,70987,70988,chr19,70200,75600,chr19:70200-75600,0.0,0.0,0.0,0.0,False
4,chr19:70988-70988,0.0,0.0,0.0,70988,70989,chr19,70200,75600,chr19:70200-75600,0.0,0.0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25998988,chr20:64334087-64334087,0.0,0.0,0.0,64334087,64334088,chr20,64330200,64335600,chr20:64330200-64335600,0.0,0.0,0.0,0.0,False
25998989,chr20:64334089-64334089,0.0,0.0,0.0,64334089,64334090,chr20,64330200,64335600,chr20:64330200-64335600,0.0,0.0,0.0,0.0,False
25998990,chr20:64334093-64334093,0.0,0.0,0.0,64334093,64334094,chr20,64330200,64335600,chr20:64330200-64335600,0.0,0.0,0.0,0.0,False
25998991,chr20:64334095-64334095,0.0,0.0,0.0,64334095,64334096,chr20,64330200,64335600,chr20:64330200-64335600,0.0,0.0,0.0,0.0,False


In [ ]:
df

,Category,Value
0,A,10
1,B,20
2,A,15
3,C,30
4,B,25
5,A,12


In [ ]:
df["A-B"] = (df["methyl_rate_ind_A"] - df["methyl_rate_ind_B"]).abs()
df["A-C"] = (df["methyl_rate_ind_A"] - df["methyl_rate_ind_C"]).abs()
df["B-C"] = (df["methyl_rate_ind_B"] - df["methyl_rate_ind_C"]).abs()
df["max_diff"] = df[["A-B","A-C","B-C"]].max(axis=1)
df["high_diff"] = False
df.loc[df["max_diff"] > 15,"high_diff"] = True

In [ ]:
# grouped.count().describe()
grouped = df[["window_id","methyl_rate_ind_A","methyl_rate_ind_B","methyl_rate_ind_C","high_diff"]].groupby("window_id")
grouped_df = grouped.mean()
grouped_df["high_diff"].value_counts(bins=20, sort=False)


(-0.001895, 0.0448]    14376
(0.0448, 0.0895]        6787
(0.0895, 0.134]          712
(0.134, 0.179]           118
(0.179, 0.224]            55
(0.224, 0.269]            22
(0.269, 0.313]            11
(0.313, 0.358]             3
(0.358, 0.403]             6
(0.403, 0.448]             2
(0.448, 0.492]             0
(0.492, 0.537]             1
(0.537, 0.582]             0
(0.582, 0.627]             2
(0.627, 0.671]             0
(0.671, 0.716]             1
(0.716, 0.761]             0
(0.761, 0.806]             0
(0.806, 0.85]              0
(0.85, 0.895]              1
Name: count, dtype: int64

In [ ]:
(grouped_df["methyl_rate_ind_A"] - grouped_df["methyl_rate_ind_B"]).value_counts(bins=20, sort=False)

(-14.369, -10.193]        1
(-10.193, -6.1]           2
(-6.1, -2.008]           64
(-2.008, 2.085]       80488
(2.085, 6.178]          459
(6.178, 10.27]           34
(10.27, 14.363]          18
(14.363, 18.456]          5
(18.456, 22.548]          0
(22.548, 26.641]          1
(26.641, 30.733]          2
(30.733, 34.826]          0
(34.826, 38.919]          1
(38.919, 43.011]          0
(43.011, 47.104]          0
(47.104, 51.197]          1
(51.197, 55.289]          0
(55.289, 59.382]          0
(59.382, 63.475]          0
(63.475, 67.567]          1
Name: count, dtype: int64

count    8.410122e+07
mean     1.313373e+00
std      4.999620e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: B-C, dtype: float64

In [ ]:
assert False

,start,end,methyl_rate,chrom,methyl_rate_2,diff
46672685,10128,10129,0.0,chr3,0.000000,0.000000
46672686,10129,10130,0.0,chr3,94.117645,-94.117645
46672687,10130,10131,0.0,chr3,0.000000,0.000000
46672688,10134,10135,0.0,chr3,0.000000,0.000000
46672689,10135,10136,50.0,chr3,5.882353,44.117647
...,...,...,...,...,...,...
84940341,198232704,198232705,0.0,chr3,NaN,NaN
84940342,198232706,198232707,0.0,chr3,NaN,NaN
84940343,198232707,198232708,0.0,chr3,NaN,NaN
84940344,198232711,198232712,0.0,chr3,NaN,NaN


In [ ]:
def group_df_by_chrom(df_to_group):
    grouped_dfs = {}
    for category, group_df in df_to_group.groupby("chrom"):
        grouped_dfs[category] = group_df
    return grouped_dfs


grouped_df1 = group_df_by_chrom(df1_small)
grouped_df2 = group_df_by_chrom(df2_small)



In [ ]:
grouped_df1.keys()

dict_keys(['chr2', 'chr3'])

In [ ]:


def extract_dfs(df1_small,chrom,count_minimum=-1,diff_threashold=-1):
    
    # df1_small = df1_small.dropna()
    # deviation_list = list(df1_small[df1_small["diff"].abs() > diff_threashold]["end"])
    ends = list(df1_small["end"])
    diffs = list(df1_small["diff"])
    # return
    if len(ends) == 0:
        return []
    curr_count = 0
    curr_start = last_one = ends[0]
    res = []
    # for end in deviation_list:
    for end,diff in zip(ends,diffs):
        if end - curr_start > 5400:
            if curr_count > count_minimum:
                # print(chrom,curr_start - 1,last_one,curr_count)
                res.append([chrom,curr_start - 1,last_one,curr_count])
            curr_start = end
            curr_count = 0
        if diff > diff_threashold:
            curr_count+=1
        last_one = end
    return res

def create_combined_df(df1_small, df2_small):
    df1_small["methyl_rate_2"] = df2_small["methyl_rate"]
    df1_small["diff"] = df1_small["methyl_rate"] - df1_small["methyl_rate_2"]
    df1_small = df1_small.dropna()
    return df1_small
res = []
for chrom in grouped_df1.keys():
    if chrom not in grouped_df2.keys():
        continue
    df1_small = create_combined_df(grouped_df1[chrom], grouped_df2[chrom])  
    res.extend(extract_dfs(df1_small,chrom,diff_threashold=DIFF_THRESHOLD))
df_results = pd.DataFrame(res,columns=["chrom","start","end","diff_count"])
df_results.diff_count.describe()

count    80231.000000
mean        48.265670
std         29.203618
min          0.000000
25%         30.000000
50%         42.000000
75%         59.000000
max       1197.000000
Name: diff_count, dtype: float64

In [ ]:
print("groups")
for category, group_df in df_results.groupby("chrom"):
    print(category)
    print(group_df["diff_count"].describe())

groups
chr2
count    44093.000000
mean        49.683941
std         31.126000
min          0.000000
25%         31.000000
50%         43.000000
75%         61.000000
max       1197.000000
Name: diff_count, dtype: float64
chr3
count    36138.000000
mean        46.535198
std         26.569135
min          0.000000
25%         30.000000
50%         41.000000
75%         57.000000
max       1054.000000
Name: diff_count, dtype: float64


In [ ]:
for key in grouped_df1.keys():
    print(key)
    df = grouped_df1[key]
    print(df["diff"].abs().describe())
    print("above threshold:",len(df[df["diff"].abs() > DIFF_THRESHOLD]),"overall:",len(df))
    

chr2
count    4.641917e+07
mean     7.563525e+00
std      2.317910e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: diff, dtype: float64
above threshold: 4485321 overall: 46672685
chr3
count    3.783507e+07
mean     7.168324e+00
std      2.255508e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+02
Name: diff, dtype: float64
above threshold: 3445932 overall: 38267661


In [ ]:
df_results.diff_count.describe()
df_results.to_csv("results/ENCFF479QRW_ENCFF713LYH_differences.csv")